# Brain Tumor 3D Segmentation with MONAI SegRegNet Model
Purpose is to train a `SegRegNet` model to preform segmentation on Brain Tumor MRI images.  The dataset used for training is available from [medical decathlon](http://medicaldecathlon.com/) and has the following characteristics:
- **Target**: Glioma segmenation necrotic/active tumor and edema
- **Modality**: FLAIR, T1w, T1gd, and T2w
- **Size**: 750 4D volumes (484 Training and 266 Testing)
- **Source**: BraTS 2016 and 2017 datasets

A paper describing the dataset can be found at [Simpson et al, 2019](https://arxiv.org/abs/1902.09063)

----
<a name='startup_tasks'></a>
## 1.0 <span style='color:blue'>|</span> Common Start Up Tasks

<a name='import_packages'></a>
### 1.1 <span style='color:blue'>|</span> Import Required Packages and Libraries

In [1]:
import os, shutil, tempfile, time, random, gc, warnings, glob
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'        # Fixes a warning from PyTorch
import torch
import pytorch_lightning as pl
import numpy as np
import matplotlib.pyplot as plt
from lightning import LightningModule, Trainer
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from typing import List, Tuple, Dict, Optional, Literal
from dataclasses import dataclass
from pathlib import Path

from monai.apps import DecathlonDataset
from monai.config import print_config
from monai.data import DataLoader, decollate_batch
from monai.handlers.utils import from_engine
from monai.losses import DiceLoss
from monai.inferers import sliding_window_inference
from monai.metrics import DiceMetric
from monai.networks.nets import SegResNet
#from monai.networks.nets import UNet
from monai.transforms import (
    Activations,
    AsDiscrete,
    Compose,
    LoadImaged,
    MapTransform,
    NormalizeIntensityd,
    Orientationd,
    RandFlipd,
    RandScaleIntensityd,
    RandShiftIntensityd,
    RandSpatialCropd,
    Spacingd,
    EnsureTyped,
    EnsureChannelFirstd,
)

from monai.utils import set_determinism
import onnxruntime
from tqdm.notebook import trange, tqdm

# Make plots have guidelines
plt.style.use('ggplot')

# Squash Python warnings
warnings.filterwarnings('ignore')

# Enable Python's Garbage Collector
gc.collect()

2026-03-02 04:03:38.018195732 [W:onnxruntime:Default, device_discovery.cc:211 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:91 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"
<frozen importlib._bootstrap_external>:1325: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
2026-03-02 04:03:39.588419: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


596

<a name='global_variables'></a>
### 1.2 <span style='color:blue'>|</span> Declare Global Variables and Set Device

In [2]:
SEED = 42
ROOT_DIR = '../monai'
MAX_EPOCHS = 50
VAL_INTERVAL = 1
VAL_AMP = True
NUM_WORKERS = 4
ROI_SIZE = [128, 128, 64]
PIX_DIM = (1.0, 1.0, 1.0)
BATCH_SIZE = 1

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

<a name='random_seed'></a>
### 1.3 <span style='color:blue'>|</span> Set Random Seed for Reproducibility
Not sure if all of these are required, but I have seen consistent results between runs

In [3]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
#set_determinism(SEED)

# When running on CuDNN backend, it is recommended to set these two options
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False
torch.set_float32_matmul_precision('medium')

----
<a name='functions_classes'></a>
## 2.0 <span style='color:blue'>|</span> Define Functions and Classes

<a name='multichannel'></a>
### 2.1 <span style='color:blue'>|</span> Convert to MultiChannel Based on `BraTS` Labels
Convert multi-class labels into multi-labels segmentation in ***One-Hot*** format

In [4]:
class ConvertToMultiChannel(MapTransform):
    '''
    Convert labels to multi channels based on brats classes:
    label 1 is the peritumoral edema
    label 2 is the GD-enhancing tumor
    label 3 is the necrotic and non-enhancing tumor core
    The possible classes are TC (Tumor core), WT (Whole tumor)
    and ET (Enhancing tumor).
    '''

    def __call__(self, data):
        d = dict(data)
        for key in self.keys:
            result = []
            # merge label 2 and label 3 to construct TC
            result.append(torch.logical_or(d[key] == 2, d[key] == 3))
            # merge labels 1, 2 and 3 to construct WT
            result.append(torch.logical_or(torch.logical_or(d[key] == 2, d[key] == 3), d[key] == 1))
            # label 2 is ET
            result.append(d[key] == 2)
            d[key] = torch.stack(result, axis=0).float()
        return d

<a name='msd_dataloader'></a>
### 2.2 <span style='color:blue'>|</span> Custom Lightning Data Module for Loading MSD Datasets

In [13]:
class msdDataLoader(pl.LightningDataModule):
    '''
    A custom LightningDataModule for loading the MSD datasets and preforming
    transforms on them
    '''

    def __init__(self,
                 root_dir: str = ROOT_DIR,
                 batch_size: int = BATCH_SIZE,
                 num_workers: int = NUM_WORKERS,
                 pix_dim = PIX_DIM,
                 seed: int = SEED):
        
        super().__init__()
        self.root_dir = root_dir
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.pix_dim = pix_dim
        self.seed = seed

        # Define train transforms
        self.train_transform = Compose(
        [
            # load 4 Nifti images and stack them together
            LoadImaged(keys=['image', 'label']),
            EnsureChannelFirstd(keys='image'),
            EnsureTyped(keys=['image', 'label']),
            ConvertToMultiChannel(keys='label'),
            Orientationd(keys=['image', 'label'], axcodes='RAS'),
            Spacingd(
                keys=['image', 'label'],
                pixdim=self.pix_dim,
                mode=('bilinear', 'nearest'),
            ),
            RandSpatialCropd(keys=['image', 'label'], roi_size=ROI_SIZE, random_size=False),
            RandFlipd(keys=['image', 'label'], prob=0.5, spatial_axis=0),
            RandFlipd(keys=['image', 'label'], prob=0.5, spatial_axis=1),
            RandFlipd(keys=['image', 'label'], prob=0.5, spatial_axis=2),
            NormalizeIntensityd(keys='image', nonzero=True, channel_wise=True),
            RandScaleIntensityd(keys='image', factors=0.1, prob=1.0),
            RandShiftIntensityd(keys='image', offsets=0.1, prob=1.0),
        ]
    )
        # Define val transforms
        self.val_transform = Compose(
            [
                LoadImaged(keys=['image', 'label']),
                EnsureChannelFirstd(keys='image'),
                EnsureTyped(keys=['image', 'label']),
                ConvertToMultiChannel(keys='label'),
                Orientationd(keys=['image', 'label'], axcodes='RAS'),
                Spacingd(
                    keys=['image', 'label'],
                    pixdim=self.pix_dim,
                    mode=('bilinear', 'nearest'),
                ),
                NormalizeIntensityd(keys='image', nonzero=True, channel_wise=True),
            ]
        )

    #
    # Prepare and setup
    #
    def prepare_data(self) -> None:
        # Called only from a single process. Use it to download / verify data.
        # In this case data is already downloaded
        pass

    def setup(self, stage: Optional[str] = None) -> None:
        # Create training dataset
        self.train_ds = DecathlonDataset(root_dir=self.root_dir, task='Task01_BrainTumour',
            transform=self.train_transform, section='training', download=False, cache_rate=0.0,
            num_workers=self.num_workers)
        
        # Define validation dataset using DecathlonDataset() as loader
        self.val_ds = DecathlonDataset(root_dir=self.root_dir, task='Task01_BrainTumour',
            transform=self.val_transform,  section='validation', download=False, cache_rate=0.0,
            num_workers=self.num_workers)
        
    def train_dataloader(self):
        return DataLoader(self.train_ds,
                          batch_size=self.batch_size,
                          shuffle=True,
                          num_workers=self.num_workers
                          )
    
    def val_dataloader(self):
        return DataLoader(self.val_ds,
                          batch_size=self.batch_size,
                          shuffle=False,
                          num_workers=self.num_workers
                          )

<a name='netork_module'></a>
### 2.3 <span style='color:blue'>|</span> Custom Lightning Module for Defining the SegResNet Model

In [6]:
class SegResNet(LightningModule):
    def __init__(self, model, lr):
        super().__init__()
        self.save_hyperperameters(ignore=['model'])
        self.model = model

        self.loss_function = DiceLoss(to_onehot_y=True, softmax=True)
        self.post_pred = Compose([EnsureType('tensor', device='cpu'), 
                                  AsDescrete(argmax=True, to_onehot=2)])
        self.post_label = Compose([EnsureType('tensor', device='cpu'), 
                                   AsDescrete(to_onehot=2)])
        self.dice_metric = DiceMetric(include_background=False, reduction='mean', get_not_nans=False)
        self.best_val_dice = 0
        self.best_val_epoch = 0
        selv.validation_step_outputs = []

    def forward(self, x):
        return self._model(x)

    def configure_optimizers(self):
        loss_function = DiceLoss(smooth_nr=0, smooth_dr=1e-5, squared_pred=True, to_onehot_y=False,
                                 sigmoid=True)
        optimizer = torch.optim.Adam(self._model.parameters(), 1e-4, weight_decay=1e-5)
        lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
        post_trans = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])

        return loss_function, optimizer, lr_scheduler, post_trans

    def training_step(self, batch, batch_idx):
        images, labels = batch['image'], batch['label']
        output = self.forward(images)
        loss = self.loss_function(output, labels)
        tensorboard_logs = {'train_loss': loss.item()}

        return {'loss': loss, 'log': tensorboard_logs}

    def validation_step(self, batch, batch_idx):
        images, labels = batch['image'], batch['label']
        roi_size = ROI_SIZE
        sw_batch_size = BATCH_SIZE
        outputs = sliding_window_inference(images, roi_size, sw_batch_size, self.forward)
        loss = self.loss_function(outputs, labels)
        outputs = [self.post_pred(i) for i in decollate_batch(outputs)]
        labels = [self.post_label(i) for i in decollate_batch(labels)]
        self.dice_metric(y_pred=outputs, y=labels)
        val_dict = {'val_loss': loss, 'val_number': len(outputs)}

        return val_dict

    def on_validation_epoch_end(self):
        val_loss, num_items = 0, 0
        for output in self.validation_step_outputs:
            val_loss += output['val_loss'].sum().item()
            num_items += output['val_number']
        mean_val_dice = self.dice_metric.agg().item()
        self.dice_metric.reset()
        mean_val_loss = torch.tensor(val_loss / num_items)
        tensorboard_logs = {'val_dice': mean_val_dice, 'val_loss': mean_val_loss}

        if mean_val_dice > self.best_val_dice:
            self.best_val_dice = mean_val_dice
            self.best_val_epoch = self.current_epoch
        print(f'current epoch: {self.current_epoch} '
              f'current mean dice: {mean_val_dice:.4f}'
              f'\nbest mean dice: {self.beatval_dice:.4f} '
              f'at epoch: {self.best_val_epoch}')

        self.validation_step_outputs.clear()
        return{'log': tensorboard_logs}           

<a name='create_model'></a>
### 2.4 <span style='color:blue'>|</span> Create SegResNet Model

In [7]:
def create_model(in_channels: int = 4, out_channels: int = 3):
    '''
    Create a SegResNet model for brain tumor segmentation

    Args:
      in_channels: Number of input channels (i.e. t1w, t1c, t2w, FLAIR)
      out_channels: Number of output labels (i.e. lable 1, 2, and 3)

    Returns:
       model: The SegResNet model object
    '''
    model = SegResNet(
        blocks_down  = [1, 2, 2, 4],
        blocks_up    = [1, 1, 1],
        init_filters = 16,
        in_channels  = in_channels,
        out_channels = out_channels,
        dropout_prob = 0.2).to(DEVICE)

    return model

<a name='create_module'></a>
### 2.5 <span style='color:blue'>|</span> Create the Data Loader Module

In [15]:
def create_module(root_dir, batch_size):
    '''
    Create a Lightning-style data loader object and separate
    out the train_loader and val_loader

    Args:
        root_dir: Root directory of the MSD dataset
        batch_size: The batch size to use when loading data
    Returns:
        data_module: The data module object
        train_dataloader:  The training dataset data loader
        val_dataloader: The validation dataset data loader
    '''
    # Create the data_module object
    data_module = msdDataLoader(root_dir=ROOT_DIR, batch_size=BATCH_SIZE)
    data_module.setup()

    # Access the train_loader() and val_loader() methods from the data_module object
    train_loader = data_module.train_dataloader()
    val_loader   = data_module.val_dataloader()

    return train_loader, val_loader, data_module

In [9]:
def train_model(data_module, num_epochs):
    '''
    Pull together resources in preparation of model training.

    Args:
        train_loader: The training dataset loader object
        val_loader:   The validation dataset loader object
    returns:
        trainer: The trainer object
    '''

    # Create SegResNet model
    model = create_model(in_channels=4, out_channels=3)

    # Create callbacks
    early_stopping = EarlyStopping(monitor='val/dice', patience=10, verbose=True,
                                   restore_best_weights=True, mode='max', strict=True)

    checkpoint = ModelCheckpoint(monitor='val/dice', mode='max', save_top_k=1,
                                 filename='best-model-{eopch:02d}-{val_dice:.2f}',
                                 verbose=False)

    # Initialize Trainer
    trainer = Trainer(
        max_epochs = MAX_EPOCHS,
        callbacks = ['early_stopping', 'checkpoint'],
        accerlerator = 'auto',
        devices = 1,
        precision = 32,
        val_check_interval = 0.25,
        gradient_clip_val = 1.0
    )

    # Train the model
    trainer.fit(model,
                train_dataloaders = train_loader,
                val_dataloaders = val_loader
                )

    return trainer    

In [14]:
data_module = create_module(root_dir=ROOT_DIR, batch_size=BATCH_SIZE)
data_module.setup()
train_loader = data_module.train_dataloader()
val_loader = data_module.val_dataloader()

In [ ]:
# Pick one image and visualize the different channels
val_data_example = val_ds[2]
print(f'image shape: {val_data_example['image'].shape}')
plt.figure('image', (24, 6))
for i in range(3):
    plt.subplot(1, 3, i + 1)
    plt.title(f'image channel {i}')
    plt.imshow(val_data_example['image'][i, :, :, 60].detach().cpu(), cmap='gray')
plt.show()

# Visualize the segmentation labels
print(f'label shape: {val_data_example['label'].shape}')
plt.figure('label', (18, 6))
for i in range(3):
    plt.subplot(1, 3, i + 1)
    plt.title(f'label channel {i}')
    plt.imshow(val_data_example['label'][i, :, :, 60].detach().cpu())
plt.show()

In [ ]:
# standard PyTorch program style: create SegResNet, DiceLoss and Adam optimizer
model = SegResNet(
    blocks_down  = [1, 2, 2, 4],
    blocks_up    = [1, 1, 1],
    init_filters = 16,
    in_channels  = 4,
    out_channels = 3,
    dropout_prob = 0.2).to(DEVICE)

loss_function = DiceLoss(smooth_nr=0, smooth_dr=1e-5, squared_pred=True, to_onehot_y=False, sigmoid=True)
optimizer = torch.optim.Adam(model.parameters(), 1e-4, weight_decay=1e-5)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)

dice_metric = DiceMetric(include_background=True, reduction='mean')
dice_metric_batch = DiceMetric(include_background=True, reduction='mean_batch')

post_trans = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])


# define inference method
def inference(input):
    def _compute(input):
        return sliding_window_inference(
            inputs = input,
            roi_size = ROI_SIZE,
            sw_batch_size = 4,
            predictor = model,
            overlap = 0.5,
        )

    if VAL_AMP:
        with torch.autocast('cuda'):
            return _compute(input)
    else:
        return _compute(input)


# use amp to accelerate training
scaler = torch.GradScaler('cuda')
# enable cuDNN benchmark
torch.backends.cudnn.benchmark = True

In [ ]:
# Early stopping parameters
patience = 10  
min_delta = 1e-4
best_metric = -1
best_metric_epoch = -1
early_stop_counter = 0
early_stop_triggered = False

best_metrics_epochs_and_time = [[], [], []]
epoch_loss_values = []
metric_values = []
metric_values_tc = []
metric_values_wt = []
metric_values_et = []

total_start = time.time()
for epoch in tqdm(range(MAX_EPOCHS), desc='Training Epochs'):
    epoch_start = time.time()
    print('-' * 10)
    print(f'epoch {epoch + 1}/{MAX_EPOCHS}')
    model.train()
    epoch_loss = 0
    step = 0
    for batch_data in train_loader:
        step_start = time.time()
        step += 1
        inputs, labels = (
            batch_data['image'].to(DEVICE),
            batch_data['label'].to(DEVICE),
        )
        optimizer.zero_grad()
        with torch.autocast('cuda'):
            outputs = model(inputs)
            loss = loss_function(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.item()

    lr_scheduler.step()
    epoch_loss /= step
    epoch_loss_values.append(epoch_loss)
    print(f'epoch {epoch + 1} average loss: {epoch_loss:.4f}')

    if (epoch + 1) % VAL_INTERVAL == 0:
        model.eval()
        with torch.no_grad():
            for val_data in val_loader:
                val_inputs, val_labels = (
                    val_data['image'].to(DEVICE),
                    val_data['label'].to(DEVICE),
                )
                val_outputs = inference(val_inputs)
                val_outputs = [post_trans(i) for i in decollate_batch(val_outputs)]
                dice_metric(y_pred=val_outputs, y=val_labels)
                dice_metric_batch(y_pred=val_outputs, y=val_labels)

            metric = dice_metric.aggregate().item()
            metric_values.append(metric)
            metric_batch = dice_metric_batch.aggregate()
            metric_tc = metric_batch[0].item()
            metric_values_tc.append(metric_tc)
            metric_wt = metric_batch[1].item()
            metric_values_wt.append(metric_wt)
            metric_et = metric_batch[2].item()
            metric_values_et.append(metric_et)
            dice_metric.reset()
            dice_metric_batch.reset()

            if metric > best_metric + min_delta:
                best_metric = metric
                best_metric_epoch = epoch + 1
                best_metrics_epochs_and_time[0].append(best_metric)
                best_metrics_epochs_and_time[1].append(best_metric_epoch)
                best_metrics_epochs_and_time[2].append(time.time() - total_start)
                torch.save(
                    model.state_dict(),
                    os.path.join(ROOT_DIR, 'best_metric_model.pth'),
                )
                print('saved new best metric model')
                early_stop_counter = 0  # reset counter on improvement
            else:
                early_stop_counter += 1  # no improvement -> increment counter

                print(f'current epoch: {epoch + 1} current mean dice: {metric:.4f}'
                    f' tc: {metric_tc:.4f} wt: {metric_wt:.4f} et: {metric_et:.4f}'
                    f'\nbest mean dice: {best_metric:.4f} at epoch: {best_metric_epoch}'
                    f'\nearly stop counter: {early_stop_counter}/{patience}'
                )

            # Check for early stopping trigger
            if early_stop_counter >= patience:
                print(f'\nEarly stopping triggered at epoch {epoch + 1}')
                early_stop_triggered = True
                break  # exit the epoch loop 

In [ ]:
def plot_results(

In [ ]:
plt.figure("train", (12, 6))
plt.subplot(1, 2, 1)
plt.title("Epoch Average Loss")
x = [i + 1 for i in range(len(epoch_loss_values))]
y = epoch_loss_values
plt.xlabel("epoch")
plt.plot(x, y, color="red")
plt.subplot(1, 2, 2)
plt.title("Val Mean Dice")
x = [VAL_INTERVAL * (i + 1) for i in range(len(metric_values))]
y = metric_values
plt.xlabel("epoch")
plt.plot(x, y, color="green")
plt.show()

plt.figure("train", (18, 6))
plt.subplot(1, 3, 1)
plt.title("Val Mean Dice TC")
x = [VAL_INTERVAL * (i + 1) for i in range(len(metric_values_tc))]
y = metric_values_tc
plt.xlabel("epoch")
plt.plot(x, y, color="blue")
plt.subplot(1, 3, 2)
plt.title("Val Mean Dice WT")
x = [VAL_INTERVAL * (i + 1) for i in range(len(metric_values_wt))]
y = metric_values_wt
plt.xlabel("epoch")
plt.plot(x, y, color="brown")
plt.subplot(1, 3, 3)
plt.title("Val Mean Dice ET")
x = [VAL_INTERVAL * (i + 1) for i in range(len(metric_values_et))]
y = metric_values_et
plt.xlabel("epoch")
plt.plot(x, y, color="purple")
plt.show()